# Cuadernillo 3 · Limpieza y separación

**Autor:** Dilan Aldana - Jeison Del Rio

**Objetivo.** Dejar los datos listos para el modelo: descartar valores imposibles, transformar las columnas y separar entrenamiento y prueba sin fuga de información.

**Fecha:** 22 de noviembre de 2026

**Desarrollo.** Cargar el archivo, revisar registros sin sentido, transformar las variables y dividir los datos. Al final, una **conclusión** con cada decisión y su motivo.

In [1]:

# Python básico
import sys
import warnings
from pathlib import Path

# Datos
import pandas as pd

# Separación
from sklearn.model_selection import train_test_split

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

warnings.filterwarnings("ignore")

print("Todo listo")


Todo listo


**Qué hace esta celda.** Carga `pandas`, `train_test_split` (la función que divide los datos) y ubica la carpeta del proyecto.

## 1. Cargar el archivo

Se lee el archivo y se crea `Aprobado`: **1 = aprobado, 0 = rechazado**.

In [2]:
df = pd.read_csv(RAIZ / "datos" / "solicitudes_prestamos.csv")
df["Aprobado"] = (df["Loan_Status"] == "Y").astype(int)

print(f"Filas: {len(df)}   Columnas: {df.shape[1]}")
print(df["Aprobado"].value_counts().rename({1: "Aprobado", 0: "Rechazado"}))

Filas: 614   Columnas: 14
Aprobado
Aprobado     422
Rechazado    192
Name: count, dtype: int64


Quedan **614 solicitudes** con **422 aprobadas y 192 rechazadas** en `Aprobado`. La tabla tiene ahora 14 columnas porque se sumó esa columna a las 13 originales.

## 2. Revisar registros sin sentido

Antes de transformar se buscan valores imposibles, que el modelo no debería aprender. La celda aplica cinco reglas:

| Regla | Qué detecta |
|---|---|
| Ingresos negativos | Alguien que gana menos de cero |
| Monto del préstamo <= 0 | Un préstamo de cero o negativo |
| Plazo <= 0 | Un plazo de cero o negativo |
| Dependientes fuera de 0 a 3 | Un número imposible de dependientes |
| `Credit_History` distinto de 0 o 1 | Un historial que no es "sí" ni "no" |

In [3]:
reglas = {
    "Ingresos negativos": df["ApplicantIncome"] < 0,
    "Monto del préstamo <= 0": df["LoanAmount"] <= 0,
    "Plazo <= 0": df["Loan_Amount_Term"] <= 0,
    "Dependientes fuera de 0 a 3": (
        ~df["Dependents"].isin(["0", "1", "2", "3+"])
        & df["Dependents"].notna()
    ),
    "Credit_History distinto de 0 o 1": (
        ~df["Credit_History"].isin([0, 1])
        & df["Credit_History"].notna()
    ),
}

pd.Series(
    {nombre: casos.sum() for nombre, casos in reglas.items()},
    name="casos",
)

Ingresos negativos                  0
Monto del préstamo <= 0             0
Plazo <= 0                          0
Dependientes fuera de 0 a 3         0
Credit_History distinto de 0 o 1    0
Name: casos, dtype: int64

Las cinco reglas dan **0 casos**: ninguna solicitud tiene un valor imposible, así que no hay que borrar ni corregir filas.

## 3. Transformar las variables

Cuatro cambios, cada uno con su motivo:

1. **Quitar `Loan_ID`:** es un código único y no ayuda a predecir; dejarlo permitiría al modelo memorizar identificadores.
2. **Convertir `"3+"` en 3:** los modelos necesitan números.
3. **Marcar los huecos de texto como `"Sin dato"`:** en vez de inventar una categoría, queda explícito que el dato falta.
4. **Unir los ingresos en `Ingreso_Total`:** para pagar la cuota importa lo que entra al hogar, no quién lo aporta.

Los huecos numéricos (`LoanAmount`, `Loan_Amount_Term`, `Credit_History`, `Dependents`) **no se rellenan aquí**: se imputan dentro del modelo (cuadernillo 4).

In [4]:
# Quitar el identificador
df = df.drop(columns="Loan_ID")

# Dependents: "3+" se convierte en 3
df["Dependents"] = df["Dependents"].replace("3+", "3").astype(float)

# Categóricas vacías: se marcan como "Sin dato"
for columna in ["Gender", "Married", "Self_Employed"]:
    df[columna] = df[columna].fillna("Sin dato")

# Una sola variable de ingreso
df["Ingreso_Total"] = df["ApplicantIncome"] + df["CoapplicantIncome"]
df = df.drop(columns=["ApplicantIncome", "CoapplicantIncome"])

print(f"Columnas ahora: {df.shape[1]}")
print("\nNulos que quedan (se imputan en el modelo):")
print(df.isna().sum()[df.isna().sum() > 0])

Columnas ahora: 12

Nulos que quedan (se imputan en el modelo):
Dependents          15
LoanAmount          22
Loan_Amount_Term    14
Credit_History      50
dtype: int64


Quedan **12 columnas**. Siguen con huecos `Dependents` (15), `LoanAmount` (22), `Loan_Amount_Term` (14) y `Credit_History` (50).

Se rellenan después de separar porque la mediana debe salir **solo de entrenamiento**: si se calculara con todo, el modelo vería información del examen antes de presentarlo (**fuga de información**) y las métricas se verían mejores de lo que son.

## 4. Separar entrenamiento y prueba

El modelo aprende con un grupo y se evalúa con otro que nunca vio; si se evaluara con los mismos datos, sería como darle el examen con las respuestas.

- **80 % para entrenar (491):** el modelo aprende de estos casos.
- **20 % para probar (123):** se guardan como examen final.

La división es **estratificada** (misma proporción de aprobados en ambos grupos) y `random_state=42` la deja fija para que cualquiera obtenga los mismos resultados.

In [5]:
y = df.pop("Aprobado")
X = pd.get_dummies(df.drop(columns="Loan_Status"), drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]} solicitudes")
print(f"Prueba:        {X_test.shape[0]} solicitudes")
print(f"Columnas para el modelo: {X.shape[1]}")
print(f"\nAprobados en entrenamiento: {y_train.mean() * 100:.1f}%")
print(f"Aprobados en prueba:        {y_test.mean() * 100:.1f}%")

Entrenamiento: 491 solicitudes
Prueba:        123 solicitudes
Columnas para el modelo: 14

Aprobados en entrenamiento: 68.6%
Aprobados en prueba:        69.1%


**491 para entrenar y 123 para probar.** En entrenamiento se aprueba el **68,6 %** y en prueba el **69,1 %**: la proporción quedó pareja y la comparación será justa.

El grupo de prueba no se usa para imputar, escalar ni elegir hiperparámetros. Solo se usa para medir (cuadernillo 4), así que las métricas reflejan datos que el modelo nunca vio.

## 5. Conclusión

- **0 registros imposibles:** las cinco reglas no encontraron casos, así que no hubo que borrar filas.
- **10 variables originales → 14 columnas:** se quitó `Loan_ID`, `"3+"` pasó a 3, los huecos de texto quedaron como `"Sin dato"` y los ingresos se unieron en `Ingreso_Total`.
- **División 491 / 123, estratificada:** 68,6 % y 69,1 % de aprobación, prácticamente iguales.
- **La imputación queda para el modelo** (cuadernillo 4), calculada solo con entrenamiento, para no contaminar la evaluación.

Con los datos limpios y separados, el cuadernillo 4 puede comparar modelos.